# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

In [3]:
class Agent:
    """An AI Agent that can use tools to help answer questions"""

    def __init__(self, model_name: str = "gpt-4o-mini", instructions: str = "Help users with any question"):
        """Initialize the agent following framework structural patterns"""
        self.model_name = model_name
        self.instructions = instructions
        self.role = "Video Game Domain Analyst"
        self.llm = LLM(model=model_name)

    def invoke(self, user_message: str) -> str:
        """Process a user message, enforce retrieval -> evaluation -> web search fallback, and return a response"""
        import json
        from lib.messages import SystemMessage, UserMessage

        print(f"\n🔍 Processing Pipeline Sequence for Input: '{user_message}'")
        
        # Step 1: Internal database query
        print(" -> Step 1: Querying local game repository data standard...")
        internal_data = get_games(num_games=3, top=True)
        
        # Step 2: Run evaluation tool
        print(" -> Step 2: Executing data accuracy verification report...")
        eval_raw = evaluate_results(retrieved_context=internal_data, user_query=user_message)
        eval_report = json.loads(eval_raw)
        
        context_to_use = internal_data
        
        # Step 3: Evaluate and conditionally trigger web search fallback
        if not eval_report.get("useful", False):
            print(f" -> Step 3 [FALLBACK TRIGGERED]: Context judged insufficient. Launching Web Search...")
            web_data = web_search(query=user_message)
            context_to_use = f"Internal Context (Insufficient):\n{internal_data}\n\nExternal Fallback Web Context:\n{web_data}"
        else:
            print(" -> Step 3: Local context determined sufficient. Skipping external search fallback.")

        # Step 4: Finalize prompt and run LLM generation
        messages = [
            SystemMessage(content=(
                f"You are an AI Agent. Your role is {self.role}. Instructions: {self.instructions}\n\n"
                f"Use the following validated reference material context to answer the user query:\n{context_to_use}"
            )),
            UserMessage(content=user_message)
        ]
        
        print(" -> Step 4: Finalizing system prompt payload synthesis and executing generation...")
        ai_response = self.llm.invoke(messages)
        
        return getattr(ai_response, 'content', str(ai_response)).strip()


In [4]:
# ==========================================
# 1. Environment & Workspace Prerequisites
# ==========================================
# ==========================================
# 1. Environment & Workspace Prerequisites
# ==========================================
import importlib.util
import sys
import re
import os
import json
import chromadb
from pathlib import Path
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from tavily import TavilyClient

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# Import workspace custom framework modules
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

# 1. Locate the exact absolute path of the .env file next to this notebook
notebook_dir = Path(os.getcwd())
env_file_path = notebook_dir / ".env"

print(f"Checking for .env file at: {env_file_path.resolve()}")

# 2. Force load via absolute path
if env_file_path.exists():
    load_dotenv(dotenv_path=env_file_path, override=True)
    print("✅ Success: .env file found and parsed.")
else:
    # Fallback to checking the parent directory just in case
    parent_env = notebook_dir.parent / ".env"
    if parent_env.exists():
        load_dotenv(dotenv_path=parent_env, override=True)
        print("✅ Success: .env file found in parent directory.")
    else:
        print("⚠️ Warning: .env file could not be detected at expected path locations.")

# 3. Guardrail validation check
openai_key = os.getenv("OPENAI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

if not openai_key or "your_" in openai_key or openai_key.strip() == "":
    raise ValueError(
        "❌ CRITICAL ERROR: 'OPENAI_API_KEY' is missing or unreadable inside your .env file! "
        "Please open your .env file in the sidebar and ensure it contains a valid token string."
    )

if not tavily_key or "your_" in tavily_key or tavily_key.strip() == "":
    raise ValueError(
        "❌ CRITICAL ERROR: 'TAVILY_API_KEY' is missing or unreadable inside your .env file!"
    )

print("🚀 Keys validated. Initializing local persistent database client connection...")

# Set up local persistent database client connection
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


# ==========================================
# 2. Pydantic Response & Validation Schemas
# ==========================================
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to completely answer the user question.")
    description: str = Field(description="Detailed explanation justifying the evaluation decision.")

class FinalAgentResponse(BaseModel):
    natural_language_summary: str = Field(description="The conversational, user-friendly response to the query.")
    structured_data: dict = Field(description="A dictionary containing supporting raw metrics, links, or facts.")


# ==========================================
# 3. Agent Tool Definitions
# ==========================================

import os
import json
from lib.tooling import tool
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage

# ==========================================
# 3. Agent Tool Definitions
# ==========================================

@tool
def query_vector_db(query_text: str, n_results: int = 3) -> str:
    """Queries the local ChromaDB vector store for relevant video game dataset details."""
    try:
        results = collection.query(
            query_texts=[query_text],
            n_results=n_results
        )
        documents = results.get("documents", [[]])
        if not documents or len(documents) == 0:
            return "No matching documents found in local database."
        return "\n\n".join([f"[Doc {i+1}]: {doc}" for i, doc in enumerate(documents)])
    except Exception as e:
        return f"Error querying vector database: {str(e)}"

@tool
def evaluate_results(retrieved_context: str, user_query: str) -> str:
    """
    Evaluates the quality and sufficiency of the retrieved internal game results 
    against the user query to decide if fallback search is required.
    """
    try:
        eval_llm = LLM(model="gpt-4o-mini")
        system_instructions = (
            "You are a helpful data validation assistant. Analyze the provided context and decide if it "
            "contains enough relevant video game details, lists, titles, or descriptions to meaningfully "
            "answer or address the user's query.\n\n"
            "If the context provides real video game repository logs relevant to the topic, mark 'useful' as true. "
            "Only return false if the context is completely blank, an error message, or entirely off-topic.\n\n"
            "You MUST return a raw JSON object matching the EvaluationReport schema:\n"
            "{\n"
            '  "useful": true/false,\n'
            '  "description": "Your brief reasoning here..."\n'
            "}"
        )
        prompt = f"User Question: {user_query}\n\nRetrieved Context:\n{retrieved_context}"
        messages = [SystemMessage(content=system_instructions), UserMessage(content=prompt)]
        raw_eval = eval_llm.invoke(messages)
        return getattr(raw_eval, 'content', str(raw_eval)).strip()
    except Exception as e:
        raise e


@tool
def web_search(query: str) -> str:
    """Performs an external web search using Tavily to find up-to-date information when internal knowledge falls short."""
    try:
        from tavily import TavilyClient
        tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))
        response = tavily.search(query=query, max_results=2)
        results = response.get("results", [])
        if not results:
            return "No relevant web search results found externally."
        return "\n".join([f"[Web]: {r['title']} - {r['content']}" for r in results])
    except Exception as e:
        return f"External web lookup failed: {str(e)}"


# ==========================================
# 4. Agent Class Implementation
# ==========================================

class UdaplayAgent(Agent):
    """
    Custom Udaplay Agent implementing strict two-tier verification:
    Internal Knowledge DB Search -> Evaluation -> Web Search Fallback.
    """
    def __init__(self, model_name: str = "gpt-4o-mini"):
        """Pass instructions to parent constructor and register tools explicitly."""
        instructions = (
            "You are the Udaplay Video Game Domain Analyst.\n\n"
            "CRITICAL WORKFLOW PROTOCOL:\n"
            "Step 1: Always execute 'query_vector_db' first to scan internal catalog records.\n"
            "Step 2: You MUST pass those results immediately into the 'evaluate_results' tool.\n"
            "Step 3: If and only if the evaluation verdict returns 'useful': false, you are "
            "permitted to fallback and run 'web_search'. Do not skip verification.\n\n"
            "Step 4: You MUST output your final response formatted strictly as a raw JSON object "
            "matching the FinalAgentResponse schema parameters."
        )
        
        super().__init__(model_name, instructions)
        
        # FIXED: Registered tools list using synchronized function references
        self.tools = [query_vector_db, evaluate_results, web_search]


# ==========================================
# 5. Agent Execution Loop & Required Queries
# ==========================================
import json

# ==========================================
# 5. Corrected Q4 Demonstration & Reporting
# ==========================================
if __name__ == "__main__":
    print("🤖 Instantiating Meeting Transcript Agent...")
    
    # Force initialize the agent to ensure it exists in this cell scope
    # Adjust this line to match your exact class constructor syntax if needed (e.g., UdaplayAgent or Agent)
    try:
        meeting_agent = UdaplayAgent(model_name="gpt-4o-mini")
    except NameError:
        try:
            meeting_agent = Agent(model_name="gpt-4o-mini")
        except TypeError:
            # Fallback if your constructor uses different default arguments
            meeting_agent = Agent()

    # SCENARIO 1: Standard Project Planning Meeting (Internal Success)
    meeting_transcript_1 = """
    Project Planning Meeting; March 15, 2024
    Attendees: John, Sarah, Mike
    Discussion:
    - Reviewed Q1 project timeline
    - Discussed resource allocation
    - Identified potential risks
    Next steps:
    1. John will update the project plan by next Friday
    2. Sarah needs to coordinate with the design team by Wednesday
    3. Mike will prepare the risk assessment document by end of month
    """

    # SCENARIO 2: Marketing Sync with Out-of-Scope Research Requirement (Triggers Web Fallback)
    meeting_transcript_2 = """
    Marketing Strategy Session; October 12, 2024
    Attendees: Alice, David, Emma
    Discussion:
    - Reviewed social media ad conversion metrics
    - Brainstormed Q4 campaign themes
    - Crucial Missing Info: We need to know what the current industry standard benchmark conversion rate is for B2B SaaS LinkedIn Ads in late 2024 to evaluate our performance.
    Next steps:
    1. David to research industry benchmark conversion rates for LinkedIn Ads online.
    2. Emma to finalize the ad copy creatives by Monday.
    """

    # SCENARIO 3: Technical Architecture Briefing (Internal Success)
    meeting_transcript_3 = """
    Technical Architecture Review; January 22, 2025
    Attendees: Alex, Priya, Carlos
    Discussion:
    - Discussed migrating database from PostgreSQL to DynamoDB
    - Analyzed latency bottlenecks in API gateway
    Next steps:
    1. Carlos will set up a local benchmark performance environment by tomorrow.
    2. Priya to draft the schema migration documentation map by Friday.
    """

    scenarios = [
        ("Scenario 1: Standard Planning Meeting", meeting_transcript_1),
        ("Scenario 2: Marketing Session (Requires External Ad Benchmarks Fallback)", meeting_transcript_2),
        ("Scenario 3: Technical Briefing", meeting_transcript_3)
    ]

    for title, transcript in scenarios:
        print(f"\n🚀 {'='*20} EXECUTING {title.upper()} {'='*20}")
        
        # Invoke the agent workflow machine safely
        run_object = meeting_agent.invoke(transcript)
        
        print("\n=======================================================")
        print(f"🎉 Execution Finished for: {title}")
        print("=======================================================")
        
        # Extract messages safely using final state hooks
        if hasattr(run_object, "get_final_state"):
            final_state = run_object.get_final_state()
            final_messages = final_state.get("messages", []) if isinstance(final_state, dict) else getattr(final_state, "messages", [])
        else:
            final_messages = [run_object] if isinstance(run_object, str) else []

        # Print out trace execution metrics
        if final_messages:
            summary_output = getattr(final_messages[-1], 'content', str(final_messages[-1])).strip()
        else:
            summary_output = str(run_object)
        
        print("\n[🤖 AGENT FINAL REASONED RESPONSE & STRUCTURAL OUTPUT]:")
        try:
            # Clean markdown code blocks if present before printing
            if summary_output.startswith("```json"):
                summary_output = summary_output.split("```json")[1].split("```")[0].strip()
            elif summary_output.startswith("```"):
                summary_output = summary_output.split("```")[1].split("```")[0].strip()
                
            parsed_summary = json.loads(summary_output)
            print(json.dumps(parsed_summary, indent=2))
        except Exception:
            print(summary_output)
            
        print("="*70)






Checking for .env file at: /workspace/Code/project/starter/.env
✅ Success: .env file found and parsed.
🚀 Keys validated. Initializing local persistent database client connection...
🤖 Instantiating Meeting Transcript Agent...

🚀 ==================== EXECUTING SCENARIO 1: STANDARD PLANNING MEETING ====================
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

🎉 Execution Finished for: Scenario 1: Standard Planning Meeting

[🤖 AGENT FINAL REASONED RESPONSE & STRUCTURAL OUTPUT]:
{
  "status": "success",
  "data": {
    "meeting_date": "March 15, 2024",
    "attendees": [
      "John",
      "Sarah",
      "Mike"
    ],
    "discussion_points": [
      "Reviewed Q1 project timeline",
     

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes